## POC

In [25]:
import requests
import json

def call_api(url_suffix, params):
    # API endpoint
    url = "https://xtracker.polymarket.com/api/" + url_suffix


    # Remove parameters that are None
    params = {k: v for k, v in params.items() if v is not None}

    # Send GET request
    response = requests.get(url, params=params)

    # Raise error if request failed
    response.raise_for_status()

    # Parse and print JSON response
    data = response.json()

    # Save to file
    with open(f"{url_suffix}.json".replace("/", "_"), "w") as f:
        json.dump(data, f, indent=2)

In [26]:
call_api(
    "users/elonmusk/trackings",
    {
        "platform": "X",
        "activeOnly": "false",
    },
)

In [22]:
call_api(
    "trackings/e2199bd0-8de4-410d-b1b2-036423385c43",
    {
        "includeStats": "true",
    },
)

In [27]:
call_api(
    "metrics/c4e2a911-36ec-4453-8a39-1edb5e6b2969",
    {
        "type": "daily",
        "startDate": "2025-02-17T16:00:00.000Z",
        "endDate": "2025-02-27T16:00:59.000Z",
    },
)

## Tweet Count Analysis

In [ ]:
from fetch_data import get_hourly_data

df, periods = get_hourly_data(force_refresh=False)
print(f"Date range: {df['date'].min()} → {df['date'].max()}")
print(f"Total hourly rows: {len(df)}")
print(f"Total tweets: {df['count'].sum()}")

Found 73 total periods, selected 28 non-overlapping
Loading cached data from /Users/user01/Documents/polymarket/elon-twit/data/hourly_tweets.csv
Date range: 2025-11-01 04:00:00+00:00 → 2026-03-13 10:00:00+00:00
Total hourly rows: 3175
Total tweets: 6404


In [28]:
import pandas as pd

daily = df.set_index("date")["count"].resample("1D").sum()
print("=== Summary Statistics ===")
print(f"  Days covered:       {len(daily)}")
print(f"  Avg tweets/day:     {daily.mean():.1f}")
print(f"  Median tweets/day:  {daily.median():.1f}")
print(f"  Max tweets/day:     {daily.max()}  ({daily.idxmax().date()})")
print(f"  Min tweets/day:     {daily.min()}  ({daily.idxmin().date()})")
print(f"  Std dev (daily):    {daily.std():.1f}")

=== Summary Statistics ===
  Days covered:       133
  Avg tweets/day:     48.2
  Median tweets/day:  43.0
  Max tweets/day:     121  (2026-01-16)
  Min tweets/day:     5  (2025-11-01)
  Std dev (daily):    25.4


In [29]:
from visualize import tweet_count_chart

tweet_count_chart(df).show()

In [30]:
from visualize import hourly_heatmap

hourly_heatmap(df).show()

In [31]:
from visualize import cumulative_chart

cumulative_chart(df, periods).show()

## Feature Engineering & Correlation

In [32]:
from features import build_hourly_features, compute_correlations

hf = build_hourly_features(df)
print(f"Hourly features: {hf.shape[0]} rows x {hf.shape[1]} columns")
hf.dropna().describe().round(2)

Hourly features: 3175 rows x 34 columns


,count,hour,day_of_week,day_of_month,is_weekend,part_of_day,lag_1h,lag_2h,lag_3h,lag_4h,...,same_hour_yesterday,same_hour_last_week,hours_since_last_tweet,day_total_so_far,consecutive_zero_hours,consecutive_active_hours,ewma_12h,ewma_24h,short_long_ratio,acceleration
count,3007.00,3007.00,3007.00,3007.00,3007.00,3007.00,3007.00,3007.00,3007.00,3007.00,...,3007.00,3007.00,3007.00,3007.00,3007.00,3007.00,3007.00,3007.00,3007.00,3007.00
mean,2.08,11.49,2.99,15.31,0.29,1.50,2.08,2.08,2.08,2.08,...,2.07,2.01,1.71,23.43,1.71,0.93,2.08,2.07,1.00,0.00
std,3.97,6.92,2.00,8.58,0.45,1.12,3.97,3.97,3.97,3.97,...,3.96,3.87,2.22,22.11,2.22,1.45,1.38,1.06,0.77,2.59
min,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.08,0.27,0.00,-12.00
25%,0.00,5.50,1.00,8.00,0.00,0.50,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,5.00,0.00,0.00,1.10,1.30,0.39,-1.33
50%,0.00,11.00,3.00,15.00,0.00,1.00,0.00,0.00,0.00,0.00,...,0.00,0.00,1.00,18.00,1.00,0.00,1.76,1.88,0.86,0.00
75%,2.00,17.00,5.00,23.00,1.00,2.00,2.00,2.00,2.00,2.00,...,2.00,2.00,3.00,34.00,3.00,1.00,2.71,2.61,1.45,1.33
max,49.00,23.00,6.00,31.00,1.00,3.00,49.00,49.00,49.00,49.00,...,49.00,49.00,17.00,121.00,17.00,11.00,10.17,6.87,4.39,11.67


In [33]:
corr = compute_correlations(hf)
print("=== Feature Correlations with Hourly Tweet Count ===")
print(corr.to_string())

import plotly.graph_objects as go

fig = go.Figure(go.Bar(
    x=corr["abs_pearson"],
    y=corr["feature"],
    orientation="h",
    marker_color=["#EF553B" if p < 0 else "#636EFA" for p in corr["pearson"]],
    text=corr["pearson"].apply(lambda x: f"{x:+.3f}"),
    textposition="outside",
))
fig.update_layout(
    title="Feature Correlation with Hourly Tweet Count (|Pearson|)",
    xaxis_title="|Pearson r|",
    template="plotly_white",
    height=max(400, len(corr) * 22),
    margin=dict(l=200),
    yaxis=dict(autorange="reversed"),
)
fig.show()

=== Feature Correlations with Hourly Tweet Count ===
                     feature  pearson  spearman  abs_pearson
0   consecutive_active_hours   0.4572    0.9183       0.4572
1     hours_since_last_tweet  -0.4024   -0.8517       0.4024
2     consecutive_zero_hours  -0.4023   -0.8521       0.4023
3                     lag_1h   0.2421    0.2749       0.2421
4            rolling_mean_2h   0.1603    0.1925       0.1603
5           rolling_mean_48h   0.1457    0.1029       0.1457
6          rolling_mean_168h   0.1305    0.0732       0.1305
7                   ewma_24h   0.1285    0.0926       0.1285
8            rolling_std_24h   0.1283    0.0749       0.1283
9                   ewma_12h   0.1283    0.0943       0.1283
10          rolling_mean_24h   0.1217    0.0808       0.1217
11           rolling_std_12h   0.1066    0.0721       0.1066
12            rolling_std_6h   0.0959    0.0543       0.0959
13          rolling_mean_12h   0.0909    0.0752       0.0909
14           rolling_mean_6h   0

## Period-Level Prediction Models

In [34]:
from fetch_data import fetch_user_trackings
from features import build_period_features

all_trackings = fetch_user_trackings()
pdf = build_period_features(df, all_trackings)
print(f"Period features: {pdf.shape[0]} rows ({pdf['period_type'].value_counts().to_dict()})")
print(f"Completed: {(~pdf['is_active']).sum()}, Active: {pdf['is_active'].sum()}")
pdf[["title", "period_type", "period_total", "pre_total_48h", "pre_mean_hourly", "prev_period_total"]].tail(15)

Period features: 69 rows ({'weekly': 39, '2day': 30})
Completed: 66, Active: 3


,title,period_type,period_total,pre_total_48h,pre_mean_hourly,prev_period_total
54,"Elon Musk # tweets February 19 - February 21, ...",2day,118,105,2.187500,105.0
55,"Elon Musk # tweets February 20 - February 27, ...",weekly,304,103,2.145833,351.0
56,"Elon Musk # tweets February 21 - February 23, ...",2day,82,114,2.375000,118.0
57,"Elon Musk # tweets February 23 - February 25, ...",2day,79,81,1.687500,82.0
58,"Elon Musk # tweets February 24 - March 3, 2026?",weekly,206,82,1.708333,304.0
59,"Elon Musk # tweets February 26 - February 28, ...",2day,50,80,1.666667,79.0
60,"Elon Musk # tweets February 27 - March 6, 2026?",weekly,210,90,1.875000,206.0
61,"Elon Musk # tweets March 2 - March 4, 2026?",2day,41,64,1.333333,50.0
62,"Elon Musk # tweets March 3 - March 10, 2026?",weekly,359,43,0.895833,210.0
63,"Elon Musk # tweets March 5 - March 7, 2026?",2day,99,85,1.770833,41.0


In [35]:
from model import train_and_evaluate, feature_importance_chart, prediction_vs_actual_chart

print("=" * 60)
info_weekly = train_and_evaluate(pdf, period_type="weekly")
print()
info_2day = train_and_evaluate(pdf, period_type="2day")

print("\n--- Results Summary ---")
for label, info in [("Weekly", info_weekly), ("2-Day", info_2day)]:
    print(f"\n{label}:")
    for r in info["results"]:
        print(f"  {r['model']:25s}  MAE={r['MAE']:6.1f}  RMSE={r['RMSE']:6.1f}  MAPE={r['MAPE%']:5.1f}%")

=== WEEKLY periods: 28 train / 7 test ===
  LinearRegression           MAE=113.4  RMSE=128.2  MAPE=44.8%
  RandomForest               MAE=72.3  RMSE=79.7  MAPE=29.1%
  GradientBoosting           MAE=74.0  RMSE=80.7  MAPE=27.1%

=== 2DAY periods: 22 train / 6 test ===
  LinearRegression           MAE=42.4  RMSE=50.0  MAPE=66.1%
  RandomForest               MAE=28.3  RMSE=37.7  MAPE=54.4%
  GradientBoosting           MAE=32.8  RMSE=39.3  MAPE=57.3%

--- Results Summary ---

Weekly:
  LinearRegression           MAE= 113.4  RMSE= 128.2  MAPE= 44.8%
  RandomForest               MAE=  72.3  RMSE=  79.7  MAPE= 29.1%
  GradientBoosting           MAE=  74.0  RMSE=  80.7  MAPE= 27.1%

2-Day:
  LinearRegression           MAE=  42.4  RMSE=  50.0  MAPE= 66.1%
  RandomForest               MAE=  28.3  RMSE=  37.7  MAPE= 54.4%
  GradientBoosting           MAE=  32.8  RMSE=  39.3  MAPE= 57.3%


In [36]:
feature_importance_chart(info_weekly, "GradientBoosting").update_layout(
    title="Weekly Period — Feature Importance (GradientBoosting)"
).show()

feature_importance_chart(info_2day, "GradientBoosting").update_layout(
    title="2-Day Period — Feature Importance (GradientBoosting)"
).show()

In [37]:
prediction_vs_actual_chart(info_weekly, "GradientBoosting").update_layout(
    title="Weekly — Predicted vs Actual"
).show()

prediction_vs_actual_chart(info_2day, "GradientBoosting").update_layout(
    title="2-Day — Predicted vs Actual"
).show()

## Trend-Change Detection

In [38]:
from model import train_trend_classifier, trend_importance_chart, trend_probability_chart

trend_info = train_trend_classifier(df)


=== Trend-Change Classifier (6h horizon) ===
  Train: 2405, Test: 602
  Accuracy: 0.625
  Label balance (test): {0: 327, 1: 275}
              precision    recall  f1-score   support

        Calm       0.65      0.66      0.66       327
       Burst       0.59      0.59      0.59       275

    accuracy                           0.62       602
   macro avg       0.62      0.62      0.62       602
weighted avg       0.62      0.62      0.62       602



In [39]:
trend_importance_chart(trend_info).show()

In [40]:
trend_probability_chart(trend_info).show()